In [8]:
# Make sure to import packages and run code from the initial_setup notebook.

# The below code will run an RBFInterpolator surrogate model using differential_evolution as an acquisition function.
# To test which kernel is the best for any particular function for RBFInterpolator, use the attached cross_validation_RBFInterpolator.ipynb. 

# Generating default random value for consistency.
rng = np.random.default_rng(19)

# Set this to the number of dimensions for whichever function you are working on. For example, for Function 5, n_dims = 4.
n_dims = 2

# In this example, we'll use Function 1. To run this analysis on a different function, simply swap it out.
# For example, insert df_function_5 instead of df_function_1.
data = df_function_1

# To use this code for other functions, add X values. For example, for Function 5, the X values would be:
# X_1, X_2, X_3, X_4 = (data[c].values for c in ['X_1','X_2','X_3','X_4'])
X_1, X_2 = (data[c].values for c in ['X_1','X_2'])

# y values stay the same for all functions
y = data['y'].values

# Aggressive output scaling plus StandardScaler for Function 1. For Functions 2-8, I used the yeo-johnson power transformation below as recommended by HEBO. 
# When running Functions 2-8, it's recommended to comment out this aggressive scaling as it will negatively impact your results.
alpha = 0.02
y_eng = np.sign(y) * np.power(np.abs(y), alpha)

scaler = StandardScaler()
y_eng = scaler.fit_transform(y_eng.reshape(-1, 1))

# Scale y values for Functions 2-8. 
# pt = PowerTransformer(method='yeo-johnson')
# y_eng = pt.fit_transform(y.reshape(-1, 1))

# Add additional X values if running other functions, similar to the above. For example, running this on Function 5
# would look like points = np.column_stack((X_1, X_2, X_3, X_4))
points = np.column_stack((X_1, X_2))

model = RBFInterpolator(
    points,
    y_eng,
    kernel='gaussian',
    epsilon=18.0,
    smoothing=1e-4
)

bounds = [(0.000000, 0.999999)] * n_dims

def neg_objective(x):
    # x arrives as a 1D array of length n; RBFInterpolator wants shape (n_points, n_dims)
    y_pred = model(x.reshape(1, -1))[0, 0]
    return -y_pred  # Differential Evolution minimizes, so negate to maximize

result = differential_evolution(
    neg_objective,
    bounds,
    strategy='best1bin',
    popsize=500, # Increase this for better forecasts. It will take longer, but not nearly as long as SMAC.
    mutation=(1.5, 1.9), # A higher mutation level plus dithering for better forecasts.
    recombination=0.1, # Lower recombination for better forecasts.
    rng=rng,
    polish=True,
    init='sobol', # sobol for the best forecasts, according to the documentation.
    updating='deferred' # For faster processing.
)

best_x = result.x
predicted_y_eng = -result.fun # Negate to maximize

# Add X values if running higher-dimensional functions
print(f"Best X_1 found: {best_x[0]:.6f}")
print(f"Best X_2 found: {best_x[1]:.6f}")

# Inverse transformations for aggressive scaling in Function 1
pred_y_undo_scaler = scaler.inverse_transform([[predicted_y_eng]])[0, 0]
pred_y_orig_scale = np.sign(pred_y_undo_scaler) * np.power(np.abs(pred_y_undo_scaler), 1 / alpha)

# Inverse transformation for yeo-johnson in Functions 2-8
# pred_y_orig_scale = pt.inverse_transform([[predicted_y_eng]])[0, 0]

print(f"Predicted objective (original scale): {pred_y_orig_scale:.6f}")

Best X_1 found: 0.378571
Best X_2 found: 0.351690
Predicted objective (original scale): 0.000141
